# Exp 030 — CoT user-state prompt + Qwen 2.5-7B (DEVSET)

**Pair-test sibling of 029.** Same retrieval (wRRF), same CoT prompt (`response_generation_cot_user_state.txt`), same `max_new_tokens=192`. Only `lm_type` changes: Qwen/Qwen2.5-1.5B-Instruct → **Qwen/Qwen2.5-7B-Instruct**.

**Why this matters**: in local smoke, Qwen 1.5B followed the structured `<user_state>` format on only 2/3 samples — a confound for the ablation. 7B should follow the format ≥95% of the time, isolating the question 'does CoT user-state lift personalization?' from 'does the small model follow CoT?'.

**Risk being tested**: prior-branch Qwen 7B + stock-prompt regressed LLM judge by −0.45 (filler words: 'fantastic', 'perfectly captures'). The new CoT prompt explicitly bans those words AND forces grounded specificity. If 7B+CoT still regresses, the bigger-model penalty is structural (not prompt-fixable) and we revert to the 1.5B path. If 7B+CoT cleanly beats 1.5B+CoT on response quality, the 7B path opens back up for the CoT-equipped pipeline.

## Hardware requirement

- **Runtime → Change runtime type → A100 40GB** (or L4 24GB).
- T4 16GB will OOM (7B bf16 ≈ 14 GB weights + KV cache). Verify with `nvidia-smi` in cell 1.

Wall time on A100: ~25-35 min for 8000 rows at batch 16.

## Read after the run

- Cell 8 prints parser-leak rates + 10 random sample responses. Compare against 029's same cells side-by-side.
- Decision gate: parser-leak <2%, format-follow rate ≥95%, response prose specific (not generic AI-speak).

In [ ]:
# 1) Verify GPU. MUST be A100 or L4 — T4 16GB will OOM on 7B bf16.
!nvidia-smi | head -20
import subprocess
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
gpu_mem_mb = int(subprocess.check_output(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits']).decode().strip())
print(f'\nGPU: {gpu_name}  ({gpu_mem_mb} MB)')
assert gpu_mem_mb >= 22000, f'Need ≥22GB GPU memory for Qwen 7B bf16; got {gpu_mem_mb} MB. Switch runtime to A100 or L4.'
print('GPU memory OK for 7B.')

In [ ]:
# 2) FORCE-FRESH clone — pull latest fresh-model code.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 2b) Mount Drive + wire persistent caches.
# Persists across Colab sessions (these are clear wins):
#   * HF datasets (talkpl-ai/* — challenge data, small, slow to download fresh)
#   * experiments/cache (BM25/dense/cf-bpr indices — 2-3 min to build)
# Does NOT persist (decided not worth it):
#   * vLLM Python install — pip --target=Drive caused too many issues
#     (--no-deps misses pieces, --upgrade silently no-ops, ~5-10 min
#     Drive writes). Local install is ~2-3 min per session, reliable.
#   * HF model weights — Drive read slower than HF download for >6 GB
#
# Drive auth: a popup appears the first time. CLICK THROUGH ALL
# permission screens — closing the popup early causes 'credential
# propagation was unsuccessful'. We retry with force_remount=True
# below if the first attempt fails.
import os, shutil
from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}\nretrying with force_remount=True ...')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)

assert os.path.isdir('/content/drive/MyDrive'), (
    'Drive mount failed — /content/drive/MyDrive does not exist. '
    'Common fixes: complete the OAuth popup fully, disable popup '
    'blocker, or sign in to Google in this browser tab on the same account.'
)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026-lora-tutorial/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

print(f'datasets cache  : {os.environ["HF_DATASETS_CACHE"]}')
print(f'experiments dir : {EXPECTED_CACHE} -> {os.readlink(EXPECTED_CACHE)}')
!ls -lh {DRIVE_BASE}/

In [ ]:
# 3) Install deps + vLLM.
# Plain local install — vLLM ends up in Colab's system site-packages.
# Reinstalls each session (~2-3 min) but reliable: no --target conflicts,
# no --no-deps gotchas, no Drive write speed issues.
!pip install -q -r requirements.txt
!pip install -q vllm

# Verify the deep import works (the test our subprocess actually exercises).
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!python -c "from vllm import LLM; import vllm; print('vllm', vllm.__version__, '-- ready')"

In [ ]:
# 4) Experiment parameters.
TID = '030-cot-user-state-qwen7b-devset'
# vLLM is configured via use_vllm: true in the yaml. It pre-allocates
# ~36 GB on A100 40GB as a shared KV cache pool (gpu_memory_utilization
# 0.9 default). 7B weights ~14 GB + KV pool ~22 GB = comfortable fit.
# BATCH_SIZE here is just a "feed cap" — vLLM continuous-batches
# internally regardless. 256 gives vLLM plenty of in-flight sequences.
BATCH_SIZE = 256
# ATTN ignored under vLLM (it picks Flash Attention 2 internally on
# Ampere+). Kept here for symmetry with sdpa fallback path.
ATTN = 'flash_attention_2'
print(f'BATCH_SIZE={BATCH_SIZE}  ATTN(ignored under vLLM)={ATTN}')
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run devset two-step inference.
#
# Env vars threaded into the subprocess:
#   PYTORCH_ALLOC_CONF=expandable_segments:True  -> reduces fragmentation
#   VLLM_USE_V1=0                                -> in-process engine
#       (avoids EngineCore subprocess CUDA init failure on Colab)
#   VLLM_WORKER_MULTIPROC_METHOD=spawn           -> avoid fork CUDA breakage
!cd music-crs-baselines && \
    PYTORCH_ALLOC_CONF=expandable_segments:True \
    VLLM_USE_V1=0 \
    VLLM_WORKER_MULTIPROC_METHOD=spawn \
    python run_inference_devset.py \
    --tid {TID} \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate prediction JSON + zip for download.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/devset/{TID}.json'
assert os.path.isfile(SRC), f'prediction not found at {SRC} — did inference fail?'

with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 8000)')
assert len(rows) >= 8000, f'only {len(rows)} rows — partial run; do not score'

stage = f'/content/_stage_{TID}'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, f'{TID}.json'))
zip_base = f'/content/{TID}'
shutil.make_archive(zip_base, 'zip', stage)
print('wrote', zip_base + '.zip')
!ls -lh {zip_base}.zip

In [ ]:
# 7a) Browser download.
from google.colab import files
files.download(f'/content/{TID}.zip')

In [ ]:
# 7b) Drive backup.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst_dir = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(f'/content/{TID}.zip', dst_dir)
shutil.copy(f'music-crs-baselines/exp/inference/devset/{TID}.json', dst_dir)
print(f'saved to Drive: {dst_dir}')
!ls -lh {dst_dir}

In [ ]:
# 8) Quality probe — sample responses + parser-leak rate.
# 7B should follow the structured format on ≥95% of rows. The parser
# (extract_cot_response in crs_baseline.py) extracts <response>...</response>
# and falls back to stripping <user_state>...</user_state> if no <response>
# tag was emitted. The FINAL parsed text going to Gemini should be near-zero
# leak (no <user_state> / <response> tags, no field names like 'mood:').
import json, random, re

with open(f'music-crs-baselines/exp/inference/devset/{TID}.json') as f:
    rows = json.load(f)

field_leak_re = re.compile(
    r'^(?:mood|intent|energy|sonic_pref|era_pref|familiarity):',
    re.M,
)
tag_leak_re = re.compile(r'<\s*/?\s*(user_state|response)\s*>', re.I)

leak_field, leak_tag, empty = 0, 0, 0
for r in rows:
    resp = (r.get('predicted_response') or '').strip()
    if not resp:
        empty += 1
        continue
    if field_leak_re.search(resp):
        leak_field += 1
    if tag_leak_re.search(resp):
        leak_tag += 1

n = len(rows)
print(f'rows total            : {n}')
print(f'empty responses       : {empty}  ({empty/n:.1%})')
print(f'field-name leak       : {leak_field}  ({leak_field/n:.1%})')
print(f'tag leak              : {leak_tag}  ({leak_tag/n:.1%})')
print()
print('=== 10 random sample responses ===')
random.seed(42)
for i in random.sample(range(n), min(10, n)):
    print(f'\n[{i}] turn {rows[i].get("turn_number")}')
    print(rows[i].get('predicted_response', '')[:400])

# Pass criteria for 7B: leak rates near 0 (parser confidence proven), no
# empty responses. If field/tag leak >2%, the parser missed cases — file an
# update to extract_cot_response.

## After the Colab run, on local M4:

```bash
cd recsys2026
TID=030-cot-user-state-qwen7b-devset
unzip -o ~/Downloads/${TID}.zip -d music-crs-baselines/exp/inference/devset/
source recsys26/bin/activate
python scripts/local_eval.py --tid ${TID} --split dev
pytest tests/test_wave2_integration.py -v
```

## Decision gate (compared to 029 / 1.5B-CoT)

- **7B response quality clearly more specific AND retrieval composite within ±0.005 of 029** → 7B+CoT is the better testbed; promote to a Blind-A candidate (still subject to fresh-model gate policy — no Blind-A ship without explicit user approval).
- **7B response quality similar to 029, no clear lift** → CoT itself is doing the work, stick with cheaper 1.5B for any Blind-A run.
- **7B response quality regresses despite cleaner format** → confirms the prior-branch 7B AI-speak failure mode is structural; 1.5B remains the production LM.